# Applied Data Science - Exercise 3

This is the third of fifth exercises in the course T-786-APDS Applied Data Science at Reykjavik University.

## Preparation

Please refer to chapters 3, 5, 6 and 7 of the textbook for the lab. Also skim over chapter 4 and make sure you are familiar with the theory.


## Task

This exercise continues from where the previous exercise left off. Having explored the car listings dataset and prepared it for machine learning algorithms, you will now train regression models in order to predict the listed car price.

You will also train classification models on the MSRP dataset.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

In [3]:
# Check python version.
# This code should run for python version >=3.6

print("python", sys.version)

python 3.12.1 (main, Dec 12 2023, 14:10:04) [Clang 13.0.0 (clang-1300.0.29.30)]


## Creating the training and test sets

We repeat the steps from the first exercise to split the data into training and test sets, making sure that we use the same `random_state` as before.

In [4]:
vehicles = pd.read_csv("../Ass1/vehicles.csv", index_col=0)

In [5]:
reduced_df = vehicles.copy()
reduced_df = reduced_df[reduced_df['price'] > 0]
reduced_df = reduced_df[reduced_df['price'].notna()]
reduced_df = reduced_df[reduced_df['price'] < 1000000]
reduced_df = reduced_df[~((reduced_df.manufacturer.isnull()) & (reduced_df.manufacturer.isnull()))]
reduced_df = reduced_df.drop(columns=[
    'county', 
    'id', 
    'region_url', 
    'url', 
    'image_url',
    'VIN'
])

In [6]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(reduced_df, test_size=0.1, random_state=123)

In [7]:
def preprocess(df_in):
    
    df = df_in.copy()
    
    N = 50
    topN_models = df.model.value_counts().head(N).index.to_numpy()
    df.loc[~df['model'].isin(topN_models),'model'] = 'unknown'
    top_models = list(train_set.model.unique())
    
    df = df.drop(columns=[
        'size', 
        'drive', 
        'lat', 
        'long', 
        'posting_date', 
        'paint_color', 
        'description',
        'state',
        'region'
    ])
    
    df.loc[~df['model'].isin(top_models),'model'] = 'unknown'
    
    df = df.dropna(subset=['year', 'odometer','manufacturer'])
    df['type'].fillna('unknown',inplace=True)
    df['title_status'].fillna('clean', inplace=True)
    df['fuel'].fillna('gas', inplace=True)
    df['cylinders'].fillna('unknown', inplace=True)
    df['transmission'].fillna('automatic', inplace=True)
    df['condition'].fillna('good', inplace=True)
    
    
    X = df.drop('price', axis=1)
    y = df['price'].copy()
    
    return(X,y)

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

X, y = preprocess(train_set)

num_attribs = list(X.select_dtypes('number'))
cat_attribs = list(X.select_dtypes('object'))


pipeline = ColumnTransformer([
    ('num', StandardScaler(), num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

X_prepared = pipeline.fit_transform(X)

/var/folders/7v/25vl4fsd1jv9jsr72bhy8wkw0000gn/T/ipykernel_21630/3941203363.py:25: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['type'].fillna('unknown',inplace=True)
/var/folders/7v/25vl4fsd1jv9jsr72bhy8wkw0000gn/T/ipykernel_21630/3941203363.py:26: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series t

## Regression models

**Task:** Train a linear regression model on the data and print the resulting mean squared error, root mean squared error, mean absolute error and mean absolute percentage error.

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

lrmodel = LinearRegression()
lrmodel.fit(X_prepared, y)

y_pred = lrmodel.predict(X_prepared)

mse = mean_squared_error(y, y_pred)
print('MSE:', mse)

rmse = np.sqrt(mse)
print('RMSE:', rmse)

mae = mean_absolute_error(y, y_pred)
print('MAE:', mae)

mape = mean_absolute_percentage_error(y, y_pred)
print('MAPE:', mape)

MSE: 136765949.83086562
RMSE: 11694.697509164811
MAE: 7575.577022145029
MAPE: 90.49611461918272


**Task:** Train a `sklearn.tree.DecisionTreeRegressor` model. First train it on the full training set and then try 3-fold cross validation.

In [ ]:
dtrmodel = DecisionTreeRegressor()
dtrmodel.fit(X_prepared, y)

y_pred = dtrmodel.predict(X_prepared)

mse = mean_squared_error(y, y_pred)
print('MSE:', mse)

rmse = np.sqrt(mse)
print('RMSE:', rmse)

mae = mean_absolute_error(y, y_pred)
print('MAE:', mae)

mape = mean_absolute_percentage_error(y, y_pred)
print('MAPE:', mape)


cross_val = cross_val_score(dtrmodel, X_prepared, y, cv=3, scoring='neg_mean_squared_error')
#Need to do negative scoring, since cross_val considers higher to be better, but we look at a lower score as better
rmse_scores = np.sqrt(-cross_val) #Flip it back to positive
print("Mean RMSE:", rmse_scores.mean())

MSE: 1051922.862548965
RMSE: 1025.6329082810112
MAE: 55.616697040064146
MAPE: 7.353525906313095
Mean RMSE: 8196.141985339988


**Question:** How does the error compare when training with/without cross validation?

**Answer:** `The error appears much lower without cross validation, but this is likely due to overfitting, since the model is being evaluated on the same data it trained on. The cross validation error is a more realistic estimate of performance on unseen data.`

**Task:** Now train a `sklearn.ensemble.RandomForestRegressor` with 3-fold cross validation. Try to find a good combination of hyperparameters by manually changing the `n_estimators` and `max_features` arguments.

In [23]:
from sklearn.ensemble import RandomForestRegressor

rfrmodel = RandomForestRegressor(n_estimators=15, max_features=50)

cross_val = cross_val_score(rfrmodel, X_prepared, y, cv=3, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-cross_val)
print("Mean RMSE:", rmse_scores.mean())

Mean RMSE: 6677.097420150084


**Question:** What was the best combination of hyperparameters you found and what was the error?

**Answer:** `The best combination found was n_estimators = 15, and max_features = 50 which resulted in a mean RMSE of 6607. This was the best combination possible without the runtime getting too long`

**Task:** Train a `RandomForestRegressor` model, this time using `sklearn.model_selection.GridSearchCV` to find a good combination of the hyperparameters `n_estimators` and `max_features`.

How long does the `GridSearchCV` take? 

*Hint*: You can use the `time` python module to record the time taken.

In [ ]:
<TODO>

**Question:** What was the best combination of hyperparameters you found using `GridSearchCV`and what was the error?

**Answer:** `<TODO>`

**Task:** Again, look for good hyperparameters for training a `RandomForestRegressor` model, but this time using `sklearn.model_selection.RandomSearchCV`.

In [ ]:
<TODO>

**Task:** Now try other regression models, including Polynomial Regression and SVM regression with different kernel types. Make sure you use cross validation and look for good hyperparameters.

In [ ]:
<TODO>

**Question:** What models did you try and what was the best hyperparameter combination for the best model? Did you get a decrease in validation error?

**Task:** Retrain your best model on the full training set. What is the resulting mean squared error, root mean squared error, mean absolute error and mean absolute percentage error?

In the upcoming lectures, you will learn about some more modeling techniques, including Deep Learning. We will be trying those techniques on our dataset as well, so we are not yet done with this training data. Thus, we will not evaluate our best model on the test data at this stage.

## Classification

Luxury cars probably have a different price distribution to other car categories. Can we distinguish between luxury cars and non-luxury cars without knowing the car's model, make and price?

We will now try classifying cars in the MSRP dataset into luxury and non-luxury categories. We will imagine that we do not have the  `model`, `make`, `msrp` and `popularity` attributes and see how accurately we can spot luxury cars.

Since the cars in the MSRP dataset are often assigned multiple market categories, we need to decide how to create a binary luxury/non-luxury variable. We will assume that the car is a luxury vehicle if one of the listed categories is `Luxury`.

In [ ]:
msrp_df = pd.read_csv("../../data/published/msrp.csv")

In [ ]:
for col in msrp_df.columns:
    new_col = '_'.join(col.lower().split(' '))
    msrp_df.rename({col: new_col}, inplace=True, axis=1)

In [ ]:
msrp_df['is_luxury'] = msrp_df.market_category.str.contains('Luxury')

In [ ]:
msrp_df = msrp_df.drop(columns=['model','market_category', 'make', 'msrp', 'popularity'])

In [ ]:
msrp_train_set, msrp_test_set = train_test_split(
    msrp_df.dropna(subset=['is_luxury']), 
    test_size=0.1, 
    random_state=123)
msrp_train_df = msrp_train_set.copy()

**Task:** Make sure you understand what the above code is doing and verify that `msrp_train_df` has the features we need.

**Task:** Get familiar with the training data, assuming now that `is_luxury` is our target variable.

**Task:** Prepare the MSRP training data for machine learning algorithms, treating `is_luxury` as the target variable.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

def preprocess_msrp(df_in):

    <TODO>
    
    return(X,y)

X, y = preprocess_msrp(msrp_train_set)

num_attribs = list(X.select_dtypes('number'))
cat_attribs = list(X.select_dtypes('object'))

pipeline = ColumnTransformer([
    ('num', StandardScaler(), num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

X_prepared = pipeline.fit_transform(X)

    

**Task:** Train binary classification models on the MSRP training data. Try different models, including Logistic Regression, Decision Tree, Random Forest, Stochastic Gradient Descent classifier and SVM. For each classifier, look at the accuracy, confusion matrix, precision, recall and F$_1$ score. When applicable, plot precision-recall curves for different thresholds and calculate the area under the curve.

In [ ]:
<TODO>

**Question:** What was the best model you found?

**Task:** Validate your best model on the MSRP test data. How does it perform?

In [ ]:
<TODO>